# 1. Imports

This section imports all libraries required for:
- data preprocessing
- feature engineering
- model training
- evaluation

### Libraries Used

#### Pandas
Used for:
- reading CSV files
- handling tabular datasets
- data cleaning and manipulation

#### NumPy
Used for:
- numerical computations
- array operations
- mathematical calculations

#### train_test_split
Used to divide the dataset into:
- training set
- validation set

This helps evaluate model performance on unseen data.

#### f1_score
Used to evaluate classification performance using:
- precision
- recall

Higher F1 score indicates better model performance.

#### mean_absolute_error
Measures average prediction error.

Lower MAE indicates better predictions.

#### LabelEncoder
Converts categorical text values into numerical values.

Example:

| Original | Encoded |
|---|---|
| Male | 1 |
| Female | 0 |

#### LGBMClassifier
LightGBM classification model used for prediction.

Advantages:
- fast training
- efficient memory usage
- strong performance on tabular datasets

In [ ]:
# 1. IMPORTS
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier

# 2. Load Data

This section loads the datasets into memory.

### Training Dataset

The training dataset contains:
- input features
- target labels (`bank_account`)

Purpose:
- train the machine learning model

### Test Dataset

The test dataset contains:
- input features only
- no target labels

Purpose:
- generate predictions for submission

### Backup Copy

A backup copy of the original test dataset is created using:

```python
original_test = test.copy()
```

Purpose:
- preserve original identifiers
- safely create the final submission file later

Especially important for:
- `uniqueid`
- `country`

In [ ]:
# 2. LOAD DATA
train = pd.read_csv("../data/Train.csv")
test = pd.read_csv("../data/Test.csv")

original_test = test.copy()

# 3. Cleaning

This section cleans the datasets before model training.

Data cleaning improves:
- model stability
- prediction quality
- consistency

### Removing Duplicate Rows

The following code removes repeated rows from the training dataset:

```python
train.drop_duplicates(inplace=True)
```

Duplicate rows may:
- bias the model
- increase overfitting
- reduce generalization performance

### Handling Missing Values

The following code replaces missing values (`NaN`) with `-1`:

```python
train.fillna(-1, inplace=True)
test.fillna(-1, inplace=True)
```

Why this is necessary:
- machine learning models cannot properly process null values
- missing values may cause training errors

Why `-1` is used:
- acts as a placeholder value
- helps the model recognize missing information

The same cleaning process is applied to:
- training dataset
- test dataset

to maintain consistency.

In [ ]:
# 3. CLEANING
train.drop_duplicates(inplace=True)

train.fillna(-1, inplace=True)
test.fillna(-1, inplace=True)

# 4. Target Processing

This section converts the target variable from text labels into numerical values.

Machine learning models require numerical targets instead of text.

### Original Target Values

| bank_account |
|---|
| Yes |
| No |

### Numerical Encoding

The following code converts text labels into numerical values:

```python
train["bank_account"] = train["bank_account"].map({"Yes": 1, "No": 0})
```

Conversion:

| Original | Encoded |
|---|---|
| Yes | 1 |
| No | 0 |

Meaning:
- `1` → customer has a bank account
- `0` → customer does not have a bank account

### Why This Step Is Important

Machine learning algorithms perform mathematical computations.

Therefore:
- text labels cannot be processed directly
- numerical representation is required

This transforms the problem into a binary classification task.

In [ ]:
# 4. TARGET
train["bank_account"] = train["bank_account"].map({"Yes": 1, "No": 0})

# 5. Combine Train and Test Data

This section combines training and test datasets into one dataframe.

Purpose:
- apply consistent preprocessing
- ensure identical feature engineering
- avoid train/test mismatch

### Creating Dataset Indicators

The following code creates a new column called `is_train`:

```python
train["is_train"] = 1
test["is_train"] = 0
```

Purpose:
- identify whether a row belongs to:
  - training dataset
  - test dataset

Values:

| Dataset | is_train |
|---|---|
| Train | 1 |
| Test | 0 |

### Adding Placeholder Target Column

The test dataset originally does not contain the target column.

To combine datasets correctly:
- both datasets must have identical columns

A temporary placeholder value `-1` is added:

```python
test["bank_account"] = -1
```

### Combining Datasets

The following code merges training and test datasets:

```python
full = pd.concat([train, test], axis=0).reset_index(drop=True)
```

Explanation:
- `pd.concat()` combines both datasets
- `axis=0` stacks rows vertically
- `reset_index(drop=True)` creates clean sequential indexing

### Why Combining Is Important

Combining datasets ensures:
- consistent encoding
- identical preprocessing
- same feature engineering logic

This helps prevent:
- train/test mismatch
- inconsistent transformations
- preprocessing errors

In [ ]:
# 5. COMBINE
train["is_train"] = 1
test["is_train"] = 0
test["bank_account"] = -1

full = pd.concat([train, test], axis=0).reset_index(drop=True)

# 6. Feature Engineering

Feature engineering creates new variables from existing features to improve model performance.

Purpose:
- extract hidden patterns
- improve prediction accuracy
- help the model learn relationships better

Feature engineering is one of the most important steps in machine learning competitions.

### Gender Encoding

The following code converts gender values into numerical format:

```python
df["is_male"] = df["gender_of_respondent"].map({"Male": 1, "Female": 0})
```

Conversion:

| Original | Encoded |
|---|---|
| Male | 1 |
| Female | 0 |

### Cellphone Access Encoding

```python
df["has_cellphone"] = df["cellphone_access"].map({"Yes": 1, "No": 0})
```

Conversion:

| Original | Encoded |
|---|---|
| Yes | 1 |
| No | 0 |

This feature may help identify financial accessibility.

### Label Encoding Categorical Features

The following categorical columns are encoded into numerical values:

```python
cat_cols = [
    "country",
    "location_type",
    "education_level",
    "job_type",
    "relationship_with_head"
]
```

Each categorical value is transformed into an integer label using `LabelEncoder`.

Purpose:
- machine learning models require numerical input
- categorical text values cannot be processed directly

### Feature Interaction

```python
df["edu_job"] = df["education_level"] * df["job_type"]
```

This creates a new feature by combining:
- education level
- job type

Purpose:
- capture relationships between multiple features
- improve model learning capability

### Age Group Binning

```python
df["age_group"] = pd.cut(...)
```

This converts continuous age values into grouped categories.

Age groups:
- 0 → children/teenagers
- 1 → young adults
- 2 → adults
- 3 → middle age
- 4 → older adults

Purpose:
- simplify age patterns
- help the model identify grouped behaviors

### Inclusion Index

```python
df["inclusion_index"] = (
    df["education_level"]
    + df["job_type"]
    + df["has_cellphone"]
)
```

This creates a combined socioeconomic indicator.

Purpose:
- summarize multiple related features
- provide stronger predictive signals

In [ ]:
# 6. FEATURE ENGINEERING
def feature_engineering(df):
    df = df.copy()

    df["is_male"] = df["gender_of_respondent"].map({"Male": 1, "Female": 0})
    df["has_cellphone"] = df["cellphone_access"].map({"Yes": 1, "No": 0})

    cat_cols = [
        "country",
        "location_type",
        "education_level",
        "job_type",
        "relationship_with_head"
    ]

    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    df["edu_job"] = df["education_level"] * df["job_type"]

    # SAFE BINNING (prevents NaN crash)
    df["age_group"] = pd.cut(
        df["age_of_respondent"],
        bins=[0, 18, 25, 35, 50, 100],
        labels=[0, 1, 2, 3, 4]
    ).astype(float).fillna(0).astype(int)

    df["inclusion_index"] = (
        df["education_level"]
        + df["job_type"]
        + df["has_cellphone"]
    )

    return df

full = feature_engineering(full)

# 7. Split Back into Train and Test

After preprocessing and feature engineering, the combined dataset is separated back into:
- training dataset
- test dataset

Purpose:
- train the model using training data
- generate predictions using test data

### Splitting Using `is_train`

The following code separates the datasets:

```python
train = full[full["is_train"] == 1].copy()
test = full[full["is_train"] == 0].copy()
```

Explanation:
- rows with `is_train = 1` belong to training data
- rows with `is_train = 0` belong to test data

`.copy()` creates independent copies of the datasets.

### Resetting Index

```python
train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)
```

Purpose:
- create clean sequential indexing
- remove old indices from the combined dataframe

This improves:
- readability
- consistency
- dataframe organization

In [ ]:
# 7. SPLIT BACK
train = full[full["is_train"] == 1].copy()
test = full[full["is_train"] == 0].copy()

train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)

# 8. Feature Selection

This section defines the input features used for machine learning.

Features are the variables the model learns from to make predictions.

### Selected Features

```python
features = [
    'country',
    'location_type',
    'education_level',
    'job_type',
    'age_of_respondent',
    'household_size',
    'is_male',
    'has_cellphone',
    'edu_job',
    'age_group',
    'inclusion_index'
]
```

These features include:
- demographic information
- socioeconomic indicators
- engineered features

### Creating Input and Target Variables

```python
X = train[features]
y = train["bank_account"]
X_test = test[features]
```

Explanation:

#### `X`
Contains training input features.

#### `y`
Contains target labels:
- 1 → has bank account
- 0 → no bank account

#### `X_test`
Contains test features used for final prediction.

Purpose:
- provide structured input to the machine learning model

In [ ]:
# 8. FEATURES
features = [
    'country',
    'location_type',
    'education_level',
    'job_type',
    'age_of_respondent',
    'household_size',
    'is_male',
    'has_cellphone',
    'edu_job',
    'age_group',
    'inclusion_index'
]

X = train[features]
y = train["bank_account"]
X_test = test[features]

# 9. Train-Validation Split

This section divides the training dataset into:
- training set
- validation set

Purpose:
- train the model on one portion
- evaluate performance on unseen data

This helps estimate how well the model generalizes.

### Dataset Split

```python
X_train, X_val, y_train, y_val = train_test_split(...)
```

Explanation:

#### `X_train`
Training features used to train the model.

#### `y_train`
Training target labels.

#### `X_val`
Validation features used for evaluation.

#### `y_val`
Validation target labels.

### Parameters Used

#### `test_size=0.2`

20% of the dataset is used for validation.

Meaning:
- 80% → training
- 20% → validation

#### `random_state=42`

Ensures reproducibility.

The same split will be generated every time the code runs.

#### `stratify=y`

Maintains the same class distribution in:
- training set
- validation set

This is important for imbalanced datasets.

In [ ]:
# 9. SPLIT
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 10. Model Training

This section initializes and trains the LightGBM classification model.

LightGBM is a gradient boosting algorithm designed for:
- speed
- efficiency
- high predictive performance

It performs especially well on structured/tabular datasets.

### Model Initialization

```python
model = LGBMClassifier(...)
```

### Parameters Used

#### `n_estimators=800`

Number of boosting trees.

Higher values may improve learning but increase training time.

#### `learning_rate=0.05`

Controls how quickly the model learns.

Smaller learning rates:
- improve stability
- reduce overfitting risk

#### `num_leaves=31`

Controls tree complexity.

Higher values:
- increase model flexibility
- may increase overfitting risk

#### `subsample=0.9`

Randomly samples rows during training.

Purpose:
- improve generalization
- reduce overfitting

#### `colsample_bytree=0.9`

Randomly samples features during tree construction.

Purpose:
- improve model robustness
- reduce feature dependency

#### `reg_alpha=0.1`

L1 regularization term.

Purpose:
- reduce unnecessary complexity
- improve stability

#### `reg_lambda=0.1`

L2 regularization term.

Purpose:
- prevent overfitting
- improve generalization

#### `random_state=42`

Ensures reproducibility of results.

#### `n_jobs=-1`

Uses all available CPU cores for faster training.

### Model Training

```python
model.fit(X_train, y_train)
```

Purpose:
- learn patterns from training data
- build predictive relationships between features and target labels

In [ ]:
# 10. MODEL (IMPROVED SAFELY)
model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 11. Model Validation

This section evaluates the model performance on unseen validation data.

We use:
- F1 Score → main competition metric
- MAE → additional error check

Validation helps detect:
- overfitting
- underfitting
- model generalization ability

In [ ]:
# 11. VALIDATION
val_preds = model.predict(X_val)

print("F1 Score:", f1_score(y_val, val_preds))
print("MAE:", mean_absolute_error(y_val, val_preds))

# 12. Final Model Training

After validation, the model is retrained on the full dataset.

This step ensures:
- maximum learning from all available data
- better final predictions for submission

In [ ]:
# 12. FINAL TRAIN
model.fit(X, y)

# 13. Prediction on Test Data

The trained model is used to predict unseen test data.

We generate:
- probabilities
- final binary predictions (0 or 1)

In [ ]:
# 13. PREDICTION
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

# 14. Submission File Creation

This section prepares the final submission file required for the competition.

It combines:
- unique IDs
- predictions

Format must match competition requirements exactly.

In [ ]:
# 14. SUBMISSION
submission_ids = (
    original_test["uniqueid"].astype(str)
    + " x "
    + original_test["country"].astype(str)
)

submission = pd.DataFrame({
    "uniqueid": submission_ids,
    "bank_account": test_preds
})

# 15. Submission Safety Check

This step ensures:
- all rows are included
- IDs are correctly formatted
- no missing predictions exist

This prevents submission errors on the competition platform.

In [ ]:
# 15. SAFETY CHECK
print(submission.head())
print("Rows:", len(submission))
print("Unique IDs:", submission["uniqueid"].nunique())

assert len(submission) == len(original_test)
assert submission["uniqueid"].nunique() == len(original_test)

print("✅ All IDs present (safe version)")

# 16. Save Submission File

This step saves the final prediction file in CSV format.

This file is what you upload to the competition platform.

Make sure:
- file name is correct
- format matches requirements

In [ ]:
# 16. SAVE
submission.to_csv("submission.csv", index=False)

print("🚀 Submission file ready!")

# Financial Inclusion Prediction Project

## Overview

This project predicts whether an individual has access to a bank account based on demographic and socioeconomic information.

The goal is to build a machine learning model that can classify users into:
- Has a bank account (1)
- Does not have a bank account (0)

The solution is designed for a structured data classification competition using feature engineering and LightGBM.

## Workflow Summary

The project follows a full machine learning pipeline:

1. Import required libraries  
2. Load training and test datasets  
3. Clean missing values and duplicates  
4. Encode target variable  
5. Combine datasets for consistent preprocessing  
6. Perform feature engineering  
7. Split data into training and validation sets  
8. Train LightGBM model  
9. Validate model performance  
10. Retrain on full dataset  
11. Generate predictions  
12. Create submission file  
13. Perform safety checks  
14. Save final output

## Data Processing

### Cleaning
- Removed duplicate rows from training data
- Filled missing values with `-1` to avoid model errors

### Target Encoding
The target variable `bank_account` was converted:
- Yes → 1  
- No → 0  

## Feature Engineering

To improve model performance, new features were created:

- **Gender encoding** → Male/Female converted to binary
- **Cellphone access** → Yes/No converted to binary
- **Label encoding** for categorical variables:
  - country
  - education level
  - job type
  - location type
  - relationship with head

### Engineered Features:
- `edu_job` → interaction between education and job type  
- `age_group` → age binned into categories  
- `inclusion_index` → combined socioeconomic score  

## Model Used

### LightGBM Classifier

A gradient boosting model used for classification tasks.

### Key Parameters:
- `n_estimators = 800`
- `learning_rate = 0.05`
- `num_leaves = 31`
- `subsample = 0.9`
- `colsample_bytree = 0.9`
- `reg_alpha = 0.1`
- `reg_lambda = 0.1`

### Why LightGBM?
- Fast training speed
- High accuracy on tabular data
- Handles large datasets efficiently
- Reduces overfitting with regularization

## Model Evaluation

The model was evaluated using:

- **F1 Score** → main competition metric
- **Mean Absolute Error (MAE)** → secondary validation metric

Validation ensures the model generalizes well to unseen data.

## Prediction Process

- Probabilities are generated using the trained model
- A threshold of 0.5 is used:
  - ≥ 0.5 → 1 (Has bank account)
  - < 0.5 → 0 (No bank account)

## Submission Format

The final submission file contains:

| uniqueid | bank_account |
|----------|-------------|
| ID x Country | 0 / 1 |

The `uniqueid` is reconstructed using:
- original ID
- country information

## Safety Checks

Before saving submission:
- Ensured all rows match original test data
- Verified unique IDs are complete
- Checked no missing predictions exist

## Output

Final output file:

```
submission27.csv
```

This file is submitted to the competition platform for evaluation.

## Tools & Libraries

- Python
- Pandas
- NumPy
- Scikit-learn
- LightGBM

## Conclusion

This project demonstrates a full machine learning pipeline including:
- data preprocessing
- feature engineering
- model training
- evaluation
- prediction generation

It is optimized for competitive machine learning performance on structured datasets.